# Webull US Market Data Probe

This notebook tests whether the Webull OpenAPI credentials in `.env` can authenticate and fetch US market data. It probes common S&P 500 ETF proxies (`SPY`, `VOO`, `IVV`) plus `AAPL` as a plain US stock sanity check, then saves any successful bar response into `data/raw/` using this project’s normal CSV schema.

In [ ]:
# Run once if the SDK is not installed in this notebook kernel.
# %pip install --upgrade webull-openapi-python-sdk pandas


In [ ]:
import logging
import os
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd

logging.disable(logging.CRITICAL)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def load_env(path: Path = ROOT / ".env") -> None:
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key.strip(), value)


def normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip().rstrip("/")
    if not endpoint:
        return ""
    parsed = urlparse(endpoint)
    if parsed.netloc:
        return parsed.netloc
    return endpoint


load_env()

required = ["WEBULL_APP_KEY", "WEBULL_APP_SECRET", "WEBULL_ACCOUNT_ID"]
missing = [key for key in required if not os.environ.get(key)]
if missing:
    raise RuntimeError(f"Missing required .env keys: {missing}")

print("Loaded Webull credentials from .env without displaying secrets.")
print("Repo root:", ROOT)


In [ ]:
from webull.core.client import ApiClient
from webull.data.common.category import Category
from webull.data.common.timespan import Timespan
from webull.data.data_client import DataClient

app_key = os.environ["WEBULL_APP_KEY"]
app_secret = os.environ["WEBULL_APP_SECRET"]

# Force US for this notebook. WEBULL_US_API_ENDPOINT may be set if Webull gave one.
endpoint = normalize_endpoint(os.environ.get("WEBULL_US_API_ENDPOINT", ""))
endpoint_candidates = [None, endpoint or None, "api.webull.com"]

auth_attempts = []
data_client = None
api_client = None
active_endpoint = None

for candidate_endpoint in dict.fromkeys(endpoint_candidates):
    try:
        client = ApiClient(app_key, app_secret, "us")
        if candidate_endpoint:
            client.add_endpoint("us", candidate_endpoint)
        candidate_data_client = DataClient(client)
    except Exception as exc:
        auth_attempts.append(
            {
                "region": "us",
                "endpoint": candidate_endpoint or "sdk-default",
                "ok": False,
                "error": repr(exc)[:240],
            }
        )
        continue

    api_client = client
    data_client = candidate_data_client
    active_endpoint = candidate_endpoint or "sdk-default"
    auth_attempts.append(
        {"region": "us", "endpoint": active_endpoint, "ok": True, "error": ""}
    )
    break

auth_attempts_df = pd.DataFrame(auth_attempts)
display(auth_attempts_df)

if data_client is None:
    raise RuntimeError(
        "Webull US SDK authentication failed. Check whether the key is enabled "
        "for US OpenAPI market data, and whether production or sandbox keys are being used."
    )

print("SDK client initialized for US market data.")
print("Active endpoint:", active_endpoint)
print("Category members:", [item.name for item in Category])
print("Timespan members:", [item.name for item in Timespan])


In [ ]:
# SPY/VOO/IVV are ETF proxies for the S&P 500. AAPL is a plain US stock sanity check.
symbols_to_probe = [
    ("SPY", Category.US_ETF.name),
    ("VOO", Category.US_ETF.name),
    ("IVV", Category.US_ETF.name),
    ("AAPL", Category.US_STOCK.name),
]
timespans = [Timespan.M5.name, Timespan.M30.name, Timespan.D.name]

probe_results = []
for symbol, category in symbols_to_probe:
    for timespan in timespans:
        try:
            response = data_client.market_data.get_history_bar(
                symbol,
                category,
                timespan,
                count="200",
            )
            status = getattr(response, "status_code", None)
            payload = response.json() if status == 200 else None
            size = len(payload) if isinstance(payload, list) else None
            probe_results.append(
                {
                    "symbol": symbol,
                    "category": category,
                    "timespan": timespan,
                    "status": status,
                    "size": size,
                    "ok": status == 200 and bool(payload),
                    "payload": payload,
                }
            )
            print(symbol, category, timespan, "status=", status, "size=", size)
        except Exception as exc:
            probe_results.append(
                {
                    "symbol": symbol,
                    "category": category,
                    "timespan": timespan,
                    "status": "exception",
                    "size": None,
                    "ok": False,
                    "error": repr(exc),
                }
            )
            print(symbol, category, timespan, "ERROR", repr(exc)[:160])

summary = pd.DataFrame(
    [{k: v for k, v in row.items() if k != "payload"} for row in probe_results]
)
summary


In [ ]:
def normalize_webull_bars(payload: object) -> pd.DataFrame:
    """Normalize likely Webull bar payloads to timestamp/open/high/low/close/volume."""
    if isinstance(payload, dict):
        for key in ("data", "bars", "items"):
            if key in payload:
                payload = payload[key]
                break
    if not isinstance(payload, list) or not payload:
        return pd.DataFrame(columns=["timestamp", "open", "high", "low", "close", "volume"])

    frame = pd.DataFrame(payload)
    rename = {}
    for source, target in {
        "time": "timestamp",
        "date": "timestamp",
        "datetime": "timestamp",
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
    }.items():
        if source in frame.columns and target not in frame.columns:
            rename[source] = target
    frame = frame.rename(columns=rename)

    if "timestamp" not in frame.columns:
        columns = frame.columns.tolist()
        raise ValueError(f"Could not identify timestamp column. Columns: {columns}")

    if pd.api.types.is_numeric_dtype(frame["timestamp"]):
        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"],
            unit="ms",
            utc=True,
        ).dt.tz_convert("America/New_York")
    else:
        frame["timestamp"] = pd.to_datetime(
            frame["timestamp"],
            utc=True,
            errors="coerce",
        ).dt.tz_convert("America/New_York")
    frame["timestamp"] = frame["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

    for col in ["open", "high", "low", "close", "volume"]:
        if col not in frame.columns:
            raise ValueError(f"Missing {col!r}. Columns: {frame.columns.tolist()}")
        frame[col] = pd.to_numeric(frame[col], errors="coerce")

    columns = ["timestamp", "open", "high", "low", "close", "volume"]
    return frame[columns].dropna().sort_values("timestamp")


successful = [row for row in probe_results if row.get("ok")]
print("Successful probes:", len(successful))
for row in successful:
    print(row["symbol"], row["category"], row["timespan"], "bars", row["size"])

successful[:1]


In [ ]:
if not successful:
    print("No successful US historical bars returned.")
else:
    # Prefer SPY 5-minute bars if available; otherwise use the first successful response.
    preferred = [
        row
        for row in successful
        if row["symbol"] == "SPY" and row["timespan"] == Timespan.M5.name
    ]
    selected = preferred[0] if preferred else successful[0]
    bars = normalize_webull_bars(selected["payload"])
    out = (
        ROOT
        / "data"
        / "raw"
        / f"webull_us_{selected['symbol'].lower()}_{selected['timespan'].lower()}.csv"
    )
    out.parent.mkdir(parents=True, exist_ok=True)
    bars.to_csv(out, index=False)
    print("Saved", len(bars), "bars to", out)
    display(bars.head())
    display(bars.tail())
